In [2]:
import numpy as np
import pandas as pd
import joblib
import os

from scipy.stats import randint
from sklearn.metrics import classification_report

from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV

from config import SEED

Elegimos dataset

In [3]:
dataset = 'crudos'

## Espectrogramas

Se suelen usar Mel espectrogramas

In [3]:
train_data = np.load(f'./dataset/ventanas_procesadas/{dataset}/train_melspectrogram.npz')
X_train = train_data['X']
y_train = train_data['y']

test_data = np.load(f'./dataset/ventanas_procesadas/{dataset}/test_melspectrogram.npz')
X_test = test_data['X']
y_test = test_data['y']

In [4]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((3993, 64128), (3993,), (999, 64128), (999,))

### Random Forest

#### Entrenamiento

In [7]:
rf = RandomForestClassifier(random_state=SEED, n_jobs=-1, max_features='sqrt')
param_distributions = {
    'max_depth': [10, 15, 20, 25],
    'min_samples_split': [10, 20, 30],
    'min_samples_leaf': [5, 10 , 15]
}

In [9]:
rnd_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_distributions,
    n_iter=20,
    scoring='roc_auc',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    verbose=3,
    random_state=SEED
)

rnd_search.fit(X_train, y_train)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
[CV 1/5] END max_depth=25, min_samples_leaf=15, min_samples_split=30;, score=0.670 total time=  12.3s
[CV 2/5] END max_depth=25, min_samples_leaf=15, min_samples_split=30;, score=0.668 total time=  15.1s
[CV 3/5] END max_depth=25, min_samples_leaf=15, min_samples_split=30;, score=0.691 total time=  14.5s
[CV 4/5] END max_depth=25, min_samples_leaf=15, min_samples_split=30;, score=0.657 total time=  14.3s
[CV 5/5] END max_depth=25, min_samples_leaf=15, min_samples_split=30;, score=0.711 total time=  14.1s
[CV 1/5] END max_depth=15, min_samples_leaf=10, min_samples_split=20;, score=0.679 total time=  17.7s
[CV 2/5] END max_depth=15, min_samples_leaf=10, min_samples_split=20;, score=0.676 total time=  24.8s
[CV 3/5] END max_depth=15, min_samples_leaf=10, min_samples_split=20;, score=0.703 total time=  16.3s
[CV 4/5] END max_depth=15, min_samples_leaf=10, min_samples_split=20;, score=0.657 total time=  25.5s
[CV 5/5] END max_dep

,estimator,RandomForestC...ndom_state=42)
,param_distributions,"{'max_depth': [10, 15, ...], 'min_samples_leaf': [5, 10, ...], 'min_samples_split': [10, 20, ...]}"
,n_iter,20
,scoring,'roc_auc'
,n_jobs,None
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,3
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [10]:
pd.DataFrame(rnd_search.cv_results_)[['param_max_depth', 'param_min_samples_leaf', 'param_min_samples_split', 'mean_test_score']].sort_values(by='mean_test_score', ascending=False).head(10)

,param_max_depth,param_min_samples_leaf,param_min_samples_split,mean_test_score
10,15,5,10,0.685927
3,25,10,10,0.685656
5,25,10,20,0.685656
6,20,10,10,0.685176
1,15,10,20,0.684012
7,15,10,10,0.684012
16,20,5,20,0.683231
2,20,15,30,0.679507
0,25,15,30,0.679424
11,25,15,20,0.679424


In [11]:
print("Best params:", rnd_search.best_params_)
print("Best CV score:", rnd_search.best_score_)

best_model = rnd_search.best_estimator_

Best params: {'min_samples_split': 10, 'min_samples_leaf': 5, 'max_depth': 15}
Best CV score: 0.6859267138539586


In [12]:
y_pred = best_model.predict(X_train)
print(classification_report(y_train, y_pred))

              precision    recall  f1-score   support

           0       1.00      0.99      0.99      1933
           1       0.99      1.00      0.99      2060

    accuracy                           0.99      3993
   macro avg       0.99      0.99      0.99      3993
weighted avg       0.99      0.99      0.99      3993



#### Evaluación

In [13]:
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.66      0.59      0.62       484
           1       0.65      0.71      0.68       515

    accuracy                           0.65       999
   macro avg       0.65      0.65      0.65       999
weighted avg       0.65      0.65      0.65       999



#### Guardado

In [14]:
os.makedirs(f'./modelos_clasicos/modelos/ventanas/{dataset}', exist_ok=True)

joblib.dump(best_model, f'./modelos_clasicos/modelos/ventanas/{dataset}/melspec_rf.pkl')

['./modelos_clasicos/modelos/ventanas/crudos/melspec_rf.pkl']

## Features de Audio

In [9]:
train_data = np.load(f'./dataset/ventanas_procesadas/{dataset}/train_features.npz')
X_train = train_data['X']
y_train = train_data['y']

test_data = np.load(f'./dataset/ventanas_procesadas/{dataset}/test_features.npz')
X_test = test_data['X']
y_test = test_data['y']

In [10]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((3993, 46), (3993,), (999, 46), (999,))

### Random Forest

#### Entrenamiento

In [ ]:
rf = RandomForestClassifier(random_state=SEED, n_jobs=-1, max_features=None)

param_distributions = {
    'max_depth': randint(10, 20),
    'min_samples_split': randint(15, 30),
    'min_samples_leaf': randint(5, 15)
}

In [6]:
rnd_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_distributions,
    n_iter=50,
    scoring='roc_auc',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    verbose=3,
    random_state=SEED
)

rnd_search.fit(X_train, y_train)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
[CV 1/5] END max_depth=16, min_samples_leaf=8, min_samples_split=27;, score=0.819 total time=   1.5s
[CV 2/5] END max_depth=16, min_samples_leaf=8, min_samples_split=27;, score=0.836 total time=   1.6s
[CV 3/5] END max_depth=16, min_samples_leaf=8, min_samples_split=27;, score=0.840 total time=   1.6s
[CV 4/5] END max_depth=16, min_samples_leaf=8, min_samples_split=27;, score=0.850 total time=   1.5s
[CV 5/5] END max_depth=16, min_samples_leaf=8, min_samples_split=27;, score=0.832 total time=   1.7s
[CV 1/5] END max_depth=17, min_samples_leaf=9, min_samples_split=21;, score=0.826 total time=   2.4s
[CV 2/5] END max_depth=17, min_samples_leaf=9, min_samples_split=21;, score=0.835 total time=   2.6s
[CV 3/5] END max_depth=17, min_samples_leaf=9, min_samples_split=21;, score=0.842 total time=   2.4s
[CV 4/5] END max_depth=17, min_samples_leaf=9, min_samples_split=21;, score=0.856 total time=   2.5s
[CV 5/5] END max_depth=17, mi

,estimator,RandomForestC...ndom_state=42)
,param_distributions,"{'max_depth': <scipy.stats....0021C60B146E0>, 'min_samples_leaf': <scipy.stats....0021C2FDF79D0>, 'min_samples_split': <scipy.stats....0021C60A07D90>}"
,n_iter,50
,scoring,'roc_auc'
,n_jobs,None
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,3
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [7]:
pd.DataFrame(rnd_search.cv_results_)[['param_max_depth', 'param_min_samples_leaf', 'param_min_samples_split', 'mean_test_score']].sort_values(by='mean_test_score', ascending=False)

,param_max_depth,param_min_samples_leaf,param_min_samples_split,mean_test_score
42,14,7,15,0.845807
14,16,6,18,0.845792
30,18,5,23,0.843806
41,12,5,17,0.842985
9,18,5,25,0.842299
19,12,5,18,0.842247
3,17,9,18,0.842134
34,12,7,15,0.841852
5,15,9,16,0.841836
22,19,8,20,0.841636


In [8]:
print("Best params:", rnd_search.best_params_)
print("Best CV score:", rnd_search.best_score_)

best_model = rnd_search.best_estimator_

Best params: {'max_depth': 14, 'min_samples_leaf': 7, 'min_samples_split': 15}
Best CV score: 0.8458065677191972


In [12]:
y_pred = best_model.predict(X_train)
print(classification_report(y_train, y_pred))

              precision    recall  f1-score   support

           0       0.98      0.95      0.96      1933
           1       0.96      0.98      0.97      2060

    accuracy                           0.97      3993
   macro avg       0.97      0.97      0.97      3993
weighted avg       0.97      0.97      0.97      3993



#### Evaluación

In [13]:
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.77      0.75      0.76       484
           1       0.77      0.79      0.78       515

    accuracy                           0.77       999
   macro avg       0.77      0.77      0.77       999
weighted avg       0.77      0.77      0.77       999



#### Guardado

In [14]:
joblib.dump(best_model, f'./modelos_clasicos/modelos/ventanas/{dataset}/features_rf.pkl')

['./modelos_clasicos/modelos/ventanas/crudos/features_rf.pkl']